# Detecção de anomalias

## Pacotes

In [0]:
%run
/CMSP/Utils/configuration

Python interpreter will be restarted.
Collecting imblearn
 Using cached imblearn-0.0-py2.py3-none-any.whl (1.9 kB)
Collecting imbalanced-learn
 Using cached imbalanced_learn-0.9.1-py3-none-any.whl (199 kB)
Requirement already satisfied: joblib>=1.0.0 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (1.0.1)
Requirement already satisfied: threadpoolctl>=2.0.0 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (2.1.0)
Collecting scikit-learn>=1.1.0
 Using cached scikit_learn-1.1.1-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (31.2 MB)
Requirement already satisfied: scipy>=1.3.2 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (1.6.2)
Requirement already satisfied: numpy>=1.17.3 in /databricks/python3/lib/python3.8/site-packages (from imbalanced-learn->imblearn) (1.19.2)
Installing collected packages: scikit-learn, imbalanced-learn, imblearn
 Attempting uninstall: scikit-learn
 Found existing installation: scikit-learn 0.24.1
 Not uninstalling scikit-learn at /databricks/python3/lib/python3.8/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-63d3a643-a183-4dfb-9299-cfd91292f6a5
 Can't uninstall 'scikit-learn'. No files were found to uninstall.
Successfully installed imbalanced-learn-0.9.1 imblearn-0.0 scikit-learn-1.1.1
WARNING: You are using pip version 21.0.1; however, version 22.1.2 is available.
You should consider upgrading via the '/local_disk0/.ephemeral_nfs/envs/pythonEnv-63d3a643-a183-4dfb-9299-cfd91292f6a5/bin/python -m pip install --upgrade pip' command.
Python interpreter will be restarted.

Out[2]: DataFrame[]

In [0]:
from IPython.display import Image
from IPython.core.display import HTML

# Methods

## zscore_filter

* A função abaixo extrai o zscore dos registro de cada variável e devolve um dataset filtrado de acordo com os desvio padroes setados pelo usuário. Assim, a função precisa receber o dataset em formato pandas, a coluna referente as classes que serão analisadas e os desvios padrões selecionados pelo analista.

### Técnica do Score Z
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.zscore.html
* Merifica a proporção de um dado em relação à sua média.
* Pode ser positivo(para a direita) ou negativo(para a esquerda)
* a proporção de 1 e -1 desvio padrões, representam 66% do conjunto total dos dados populacionais
* a proporção de 2 e -2 desvio padrões, representam 95% do conjunto total dos dados populacionais
* a proporção de 3 e -3 desvio padrões, representam 99,7% do conjunto total dos dados populacionais

### Exemplos Gráficos

In [0]:
Image(url='https://i1.wp.com/makemeanalyst.com/wp-content/uploads/2017/05/Z-Score-and-Emperical-rule.png?resize=676%2C345')

Out[47]:

### Função

In [0]:
def zscore_filter(dados, target, dvpd, ddof=0, nan_policy='propagate'):
  # https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/2194656763499793/command/2194656763500011
  '''
  dados = dataset em formato pandas.
  target = coluna de identificação da variável target. Apenas se forem classes.
  dvpd = numero de desvio padrão que o modelo assumirá
  retorno = identificação de outliers dentro do conjunto de dados  e os elimina do dataset.
  '''
  dados = dados.toPandas()
  tg_list = dados[target].unique() #* verificar as classes distintas
  name_col = list(dados.drop(target, axis=1)) #* selecionar a coluna
  filtro = dados.columns
  df_retorno = []
  for values, tg in enumerate(tg_list):
    subset = dados[dados[target] == tg] # quebar o dataset por classe em outros df
    lista_outlier = []
    for variavel in name_col:
      X = np.array(subset[variavel])
      out = zscore(X, axis=0, ddof=ddof, nan_policy=nan_policy)
      subset[f'{variavel}_outlier'] = out
      lista_outlier.append(f'{variavel}_outlier')
      for i in lista_outlier:
        subset = subset[subset[i] <= dvpd]
        subset = subset[subset[i] >= ((dvpd)* -1)]
    df_retorno.append(subset)
  retorno = pd.concat(df_retorno)
  retorno = retorno[filtro]
  retorno = spark.createDataFrame(retorno)
  return retorno